In [1]:
import pandas as pd
from collections import defaultdict
#import time
#from datetime
import datetime

def lectura_xls(file_path):
    """
    Lee el archivo Excel y extrae los campos y reglas de calidad.
    
    Args:
        file_path (str): Ruta del archivo Excel.
    
    Returns:
        tuple: (active_dimensions, reglas_dict)
    """
    with pd.ExcelFile(file_path) as xls:
        # Leer nombres de campos de pestañas GIF y TAXONOMIA
        #gif_fields = pd.read_excel(xls, sheet_name='GIF').columns.tolist()
        #lider_adscripcion_fields = pd.read_excel(xls, sheet_name='LIDER_ADSCRIPCION').columns.tolist()
        #taxonomia_fields = pd.read_excel(xls, sheet_name='TAXONOMIA').columns.tolist()
        #taxonomia_hist = pd.read_excel(xls, sheet_name='TAXONOMIA_HIST').columns.tolist()
        
               
        # Leer pestaña REGLAS_DE_CALIDAD y rellenar valores vacíos con 0
        reglas_df = pd.read_excel(xls, sheet_name='REGLAS_DE_CALIDAD').fillna(0)
    
    # Crear un diccionario para mapear (fuente, campo) a dimensiones activas
    reglas_dict = {}
    for _, row in reglas_df.iterrows():
        fuente = row['ORIGEN']
        campo = row['NOMBRE_CAMPO']
        key = (fuente, campo)
        reglas_dict[key] = {
            'INTEGRIDAD': int(row.get('INTEGRIDAD', 0)),
            'COMPLETITUD': int(row.get('COMPLETITUD', 0)),
            'UNICIDAD': int(row.get('UNICIDAD', 0)),
            'CONSISTENCIA': int(row.get('CONSISTENCIA', 0)),
            'VALIDEZ_1': int(row.get('VALIDEZ_1', 0)),
            'VALIDEZ_2': int(row.get('VALIDEZ_2', 0)),
            'OPORTUNIDAD': int(row.get('OPORTUNIDAD', 0)),
        }
    
    # Crear active_dimensions: por cada dimension, lista de (fuente, campo) que la tienen activa
    active_dimensions = defaultdict(list)
    for (fuente, campo), dims in reglas_dict.items():
        for dim in dims:
            if dims[dim] == 1:
                active_dimensions[dim].append((fuente, campo))
    
    return dict(active_dimensions), reglas_dict

# Función para validar el formato de una fecha
def validar_fecha(valor):
    # Definir formatos válidos
    #date_formats = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%Y.%m.%d']              
    date_formats = [
        # Solo fecha
        '%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%Y.%m.%d',
        # Fecha + hora
        '%Y-%m-%d %H:%M', '%Y/%m/%d %H:%M', '%d/%m/%Y %H:%M', '%m/%d/%Y %H:%M',
        # Fecha + hora + segundos
        '%Y-%m-%d %H:%M:%S', '%Y/%m/%d %H:%M:%S', '%d/%m/%Y %H:%M:%S', '%m/%d/%Y %H:%M:%S',
        # Fecha + hora con microsegundos
        '%Y-%m-%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S.%f', '%d/%m/%Y %H:%M:%S.%f', '%m/%d/%Y %H:%M:%S.%f',
        # Formatos ISO extendidos
        '%Y-%m-%dT%H:%M:%S', '%Y-%m-%dT%H:%M:%S.%f'
    ]
    for fmt in date_formats:
        try:
            fecha = datetime.datetime.strptime(str(valor).strip(), fmt)
                    #datetime.datetime.strptime(str(valor), fmt)
            if 1979 <= fecha.year <= 2030:
                return True  # Formato válido
            else:
                return False  # Fuera del rango permitido
        except (ValueError, TypeError):
            continue
    return False  # Ningún formato válido

def validar_datos(file_path):
    """
    Valida las dimensiones de calidad de datos y genera archivos de salida.
    
    Args:
        file_path (str): Ruta del archivo Excel de entrada.
    """
    active_dimensions, reglas_dict = lectura_xls(file_path)
    
    # Archivos de salida
    output_consolidado = f'Resultados_Perfilamiento/Perfilamiento_Consolidado.xlsx'
    output_sheet_template = 'Resultados_Perfilamiento/Perfilamiento_{}.xlsx'
    
    consolidado_data = []
    invalidos_detalles = defaultdict(list)

    taxonomia_df = pd.read_excel(file_path, sheet_name='TAXONOMIA')
    taxonomia_nominas = set(taxonomia_df['NOMINA_PROFESOR'].dropna())

    #print(f' Total profesores taxonomía: {len(taxonomia_nominas)}')

    #taxonomia_hist_df = pd.read_excel(file_path, sheet_name='TAXONOMIA_HIST')
    #taxonomia_nominas_hist = set(taxonomia_hist_df['NOMINA_PROFESOR'].dropna())

    all_sheets = pd.ExcelFile(file_path).sheet_names
    #sheets_to_process = [s for s in all_sheets if s != 'TAXONOMIA' and s != 'TAXONOMIA_HIST' and s != 'REGLAS_DE_CALIDAD']
    sheets_to_process = [s for s in all_sheets if s != 'REGLAS_DE_CALIDAD']

    
    # Procesar cada pestaña
    #
    for sheet_name in sheets_to_process:
        print(f"Procesando pestaña {sheet_name}...")
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        total_registros = len(df)

        #almacenar el detalle de incidentes por cada pestaña
        invalidos_detallados = []
        # Diccionario para almacenar contadores por campo y dimensión
        campo_dim_counts = defaultdict(lambda: defaultdict(int))
        
        for campo in df.columns:
            key = (sheet_name, campo)
            if key not in reglas_dict:
                continue
            dims_activas = reglas_dict[key]
            
            # 1. Integridad: INCLUIR LAS COLUMNAS DONDE APLIQUE LA INTEGRIDAD
            #if sheet_name in('LIDER_ADSCRIPCION','GRADOS_ACADEMICOS','LISTADO_MATERIAS','NIVEL_IMPARTICION') and campo == 'NOMINA_PROFESOR' and dims_activas.get('INTEGRIDAD', 0) == 1:
            #if sheet_name in('GIF') and campo == 'NOMINA_PROFESOR' and dims_activas.get('INTEGRIDAD', 0) == 1:

           
            if campo == 'NOMINA_FACULTY' and dims_activas.get('INTEGRIDAD', 0) == 1:
                
                #obtener profesores únicos para integridad
                #lista_profesores_integridad = df[campo].dropna().nunique()
                #print(lista_profesores_integridad)

                #obtener profesores de taxonomía al 25 oct 25
                profesores_analizados=df[campo].dropna().nunique()
                lista_profesores_integridad = len(taxonomia_nominas)
                
                # Crear máscara para valores distintos a la taxonomía y no nulos (al día y el histórico)             
                mask = ~df[campo].isin(taxonomia_nominas) & df[campo].notna()
                #mask_hist = ~df[campo].isin(taxonomia_nominas_hist) & df[campo].notna()
                
                # Contabilizar cuántos valores no cumplen la taxonomía al día + histórico
                campo_dim_counts[campo]['INTEGRIDAD'] = int(mask.sum())
                #campo_dim_counts[campo]['INTEGRIDAD_HIST'] = int(mask_hist.sum())

                # Extraer los valores únicos que no están en la taxonomía
                valores_diferentes_nomina = df.loc[mask, campo].unique().tolist()
                #valores_diferentes_nomina_hist = df.loc[mask_hist, campo].unique().tolist()
                
                #Registro de detalle de las incidencias de  INTEGRIDAD TAXONOMÍA AL DÍA
                for idx, val in df.loc[mask, campo].items():
                    invalidos_detallados.append({
                    'Fuente de Información': sheet_name,
                    'Nómina profesor': val,
                    'Campo': campo,
                    'Valor': val,
                    'Dimensión': 'INTEGRIDAD',
                    'Regla': 'El profesor no existe en taxonomía del último corte',
                    'Descripción': 'Profesor no encontrado.',
                    'Registro': idx + 2
                    })
                #Registro de detalle de las incidencias de  INTEGRIDAD HISTÓRICO
                #for idx, val in df.loc[mask_hist, campo].items():
                 #   invalidos_detallados.append({
                  #  'Fuente de Información': sheet_name,
                   # 'Nómina profesor': val,
                    #'Campo': campo,
                    #'Valor': val,
                    #'Dimensión': 'INTEGRIDAD_HIST',
                    #'Regla': 'El profesor no existe en Taxonomía histórica',
                    #'Descripción': 'Profesor no encontrado.',
                    #'Registro': idx + 2
                    #})
            else:
                 profesores_analizados=0
                
            # 2. Completitud
            if dims_activas.get('COMPLETITUD', 0) == 1:
                mask = df[campo].isnull()
                campo_dim_counts[campo]['COMPLETITUD'] = int(mask.sum())
                for idx, val in df.loc[mask, campo].items():
                    nomina = df.loc[idx, 'NOMINA_FACULTY'] #if sheet_name == 'GIF' else ''
                    invalidos_detallados.append({
                    'Fuente de Información': sheet_name,
                    'Nómina profesor' :nomina,    
                    'Campo': campo,
                    'Valor': val,
                    'Dimensión': 'COMPLETITUD',
                    'Regla': 'El campo no debe estar vacío',
                    'Descripción': 'Campo vacío',
                    'Registro': idx + 2
                    })
            
            # 3. Unicidad
            if dims_activas.get('UNICIDAD', 0) == 1:
                if total_registros == 0:
                    campo_dim_counts[campo]['UNICIDAD'] = 0
                else:
                    mask = df[campo].duplicated(keep=False)
                    campo_dim_counts[campo]['UNICIDAD'] = int(mask.sum())
                    for idx, val in df.loc[mask, campo].items():
                        nomina = df.loc[idx, 'NOMINA_FACULTY']
                        invalidos_detallados.append({
                        'Fuente de Información': sheet_name,
                        'Nómina profesor': nomina,
                        'Campo': campo,
                        'Valor': val,
                        'Dimensión': 'UNICIDAD',
                        'Regla': 'No deben existir nóminas duplicadas',
                        'Descripción': 'Nómina duplicada',
                        'Registro': idx + 2
                    })
            
            # 4. Consistencia (siempre válido)
            if dims_activas.get('CONSISTENCIA', 0) == 1:
                campo_dim_counts[campo]['CONSISTENCIA'] = 0  # No hay invalidos
            
            # 5. Validez_1 (regex)
            if dims_activas.get('VALIDEZ_1', 0) == 1: #and campo != 'NOMINA_PROFESOR':
                # Asumir que es texto
                pattern = r'^[A-Za-z0-9ñÑáéíóúÁÉÍÓÚüÜ\'@._, ()ö´ãâä-]+$'
                mask = ~df[campo].astype(str).str.fullmatch(pattern, case=False)
                campo_dim_counts[campo]['VALIDEZ_1'] = int(mask.sum())
                for idx, val in df.loc[mask, campo].items():
                    nomina = df.loc[idx, 'NOMINA_FACULTY']
                    invalidos_detallados.append({
                    'Fuente de Información': sheet_name,
                    'Nómina profesor': nomina,
                    'Campo': campo,
                    'Valor': val,
                    'Dimensión': 'VALIDEZ_1',
                    'Regla': 'Sólo se permiten caracteres alfanuméricos, espacios y símbolos básicos',
                    'Descripción': 'Caracteres no permitidos',
                    'Registro': idx + 2
                    })
            
            # 6. Validez_2 (numérico >=0 o fecha válida)
            if dims_activas.get('VALIDEZ_2', 0) == 1:
                mask_validez2 = pd.Series([False]*len(df), index=df.index)

                if campo != "NOMINA_PROFESOR":
                    # Intentar como numérico            
                    if pd.api.types.is_numeric_dtype(df[campo]):
                        mask_validez2 |= (df[campo] < 0)
                    else:
                        # Aplicar validación por fila
                        mask_validez2 = df[campo].apply(lambda x: not validar_fecha(x) if pd.notna(x) else False)
                        campo_dim_counts[campo]['VALIDEZ_2'] = int(mask_validez2.sum())
                
                for idx, val in df.loc[mask_validez2, campo].items():
                    nomina = df.loc[idx, 'NOMINA_FACULTY']
                    invalidos_detallados.append({
                    'Fuente de Información': sheet_name,
                    'Nómina profesor': nomina,
                    'Campo': campo,
                    'Valor': val,
                    'Dimensión': 'VALIDEZ_2',
                    'Regla': 'Validación de fechas y rangos de edad',
                    'Descripción': 'Caracteres no permitidos',
                    'Registro': idx + 2
                    })
                
            
            # 7. Oportunidad
            if dims_activas.get('OPORTUNIDAD', 0) == 1:
                if campo != "NOMINA_PROFESOR":
                    mask = ~df[campo].astype(str).str.contains('2025', na=False) #startswith('2025', na=False)
                    campo_dim_counts[campo]['OPORTUNIDAD'] = int((~mask).sum())
                
                    for idx, val in df.loc[mask, campo].items():
                        nomina = df.loc[idx, 'NOMINA_FACULTY']
                        invalidos_detallados.append({
                        'Fuente de Información': sheet_name,
                        'Nómina profesor': nomina,
                        'Campo': campo,
                        'Valor': val,
                        'Dimensión': 'OPORTUNIDAD',
                        'Regla': 'El periodo debe comenzar con "2025"',
                        'Descripción': 'Periodo fuera de 2025',
                        'Registro': idx + 2
                    })
            
            # Conteo de registros con datos
            registros_con_datos = int(df[campo].count())

            # Contabilizar total profesores para integridad
            total_profesores = lista_profesores_integridad if campo == 'NOMINA_FACULTY' else 0
                #len(lista_profesores_integridad) if campo == 'NOMINA_FACULTY' else 0
                         
            # Dimensiones
            #conteo_integridad = campo_dim_counts[campo].get('INTEGRIDAD', 0)
            #conteo_integridad = len(valores_diferentes_nomina) if campo == 'NOMINA_FACULTY' else 0 cambio 2910 oct
            conteo_integridad = profesores_analizados if campo == 'NOMINA_FACULTY' else 0 # cambio 2910 oct 
            #conteo_integridad_hist = len(valores_diferentes_nomina_hist) if campo == 'NOMINA_FACULTY' else 0
            conteo_completitud = campo_dim_counts[campo].get('COMPLETITUD', 0)
            conteo_unicidad = campo_dim_counts[campo].get('UNICIDAD', 0)
            conteo_consistencia = campo_dim_counts[campo].get('CONSISTENCIA', 0)
            conteo_validez_1 = campo_dim_counts[campo].get('VALIDEZ_1', 0)
            conteo_validez_2 = campo_dim_counts[campo].get('VALIDEZ_2', 0)
            conteo_oportunidad = campo_dim_counts[campo].get('OPORTUNIDAD', 0)
            
            # Calcular DQ
            if total_registros == 0:
                dq_integridad = 0
                #dq_integridad_hist = 0
                dq_completitud = 0
                dq_unicidad = 0
                dq_consistencia = 0
                dq_validez = 0
                dq_oportunidad = 0
            else:
                #dq_integridad = 100 * (1 - conteo_integridad / total_profesores)
                if total_profesores > 0 and conteo_integridad > 0:
                    dq_integridad = 100 * (conteo_integridad / total_profesores)
                    #dq_integridad = 100 * (1 - conteo_integridad / total_profesores)
                else:
                    dq_integridad = 0  # o 100, según tu criterio de negocio
                    
                #if total_profesores > 0 and conteo_integridad_hist > 0:
                #    dq_integridad_hist = 100 * (1 - conteo_integridad_hist / total_profesores)
                #else:
                 #   dq_integridad_hist = 0  # o 100, según tu criterio de negocio                    

            dq_completitud = 100 * (1 - conteo_completitud / total_registros)
            dq_unicidad = 100 * (1 - conteo_unicidad / total_registros)
            dq_consistencia = 100 * (1 - conteo_consistencia / total_registros)
            dq_validez = 100 * (1 - (conteo_validez_1 + conteo_validez_2) / total_registros)
            dq_oportunidad = 100 * (1 - conteo_oportunidad / total_registros)
            
            valores_dq = [
            dq_integridad,
            #dq_integridad_hist,
            dq_completitud,
            dq_unicidad,
            dq_consistencia,
            dq_validez,
            dq_oportunidad
            ]

            # Filtrar solo valores > 0
            valores_validos = [v for v in valores_dq if v > 0]

            # Calcular promedio solo si hay valores válidos
            dq_calidad_total = sum(valores_validos) / len(valores_validos) if valores_validos else 0
            #dq_calidad_total = (dq_integridad + dq_completitud + dq_unicidad + dq_consistencia + dq_validez + dq_oportunidad) / 6
            
            consolidado_data.append({
                'llave': f"{sheet_name}_{campo}",
                'fuente de información': sheet_name,
                'campo': campo,
                'Total de registros': total_registros,
                'Total de registros con datos': registros_con_datos,
                'Total de profesores Taxonomía': total_profesores,
                #'Total de profesores analizados': profesores_analizados,
                'Integridad': conteo_integridad,
                #'Integridad_hist': conteo_integridad_hist,
                'completitud': conteo_completitud,
                'Unicidad': conteo_unicidad,
                'Consistencia': conteo_consistencia,
                'Validez': conteo_validez_1 + conteo_validez_2,
                'oportunidad': conteo_oportunidad,
                'DQ_Integridad': dq_integridad,
                #'DQ_Integridad_hist': dq_integridad_hist,
                'DQ_Completitud': dq_completitud,
                'DQ_Unicidad': dq_unicidad,
                'DQ_Consistencia': dq_consistencia,
                'DQ_Validez': dq_validez,
                'DQ_Oportunidad': dq_oportunidad,
                'DQ_Calidad_Total': dq_calidad_total,
            })
            
            # Agregar datos para el archivo por pestaña
            invalidos_detalles[sheet_name].append({
                'llave': f"{sheet_name}_{campo}",
                'fuente de información': sheet_name,
                'campo': campo,
                'valor': 'Valores invalidos',
                'INTEGRIDAD': conteo_integridad,
                #'INTEGRIDAD_HIST': conteo_integridad_hist,
                'COMPLETITUD': conteo_completitud,
                'UNICIDAD': conteo_unicidad,
                'CONSISTENCIA': conteo_consistencia,
                'VALIDEZ_1': conteo_validez_1,
                'VALIDEZ_2': conteo_validez_2,
                'OPORTUNIDAD': conteo_oportunidad,
            })

        # Guardar archivo detallado de datos inválidos
        df_invalidos_detallados = pd.DataFrame(invalidos_detallados)
        
        # Guardar archivo detallado de datos inválidos como CSV con separador tabulación (TSV)
        output_detail_path = f'Resultados_Perfilamiento/Datos_Invalidos_Detallados_{sheet_name}.tsv'
        
        df_invalidos_detallados.to_csv(
                               output_detail_path,
                               sep='\t',
                               index=False, 
                               encoding='cp1252')  # utf-8-sig para acentos en Excel
        print("Archivo de datos inválidos detallados guardado.")
    # Crear DataFrame consolidado
    df_consolidado = pd.DataFrame(consolidado_data)
    
    # Guardar archivo consolidado
    df_consolidado.to_excel(output_consolidado, index=False)
    print(f"Archivo consolidado guardado en {output_consolidado}")
    
    # Guardar archivos por pestaña
    for sheet_name, records in invalidos_detalles.items():
        df_invalidos = pd.DataFrame(records)
        output_file = output_sheet_template.format(sheet_name)
        df_invalidos.to_excel(output_file, index=False)
        print(f"Archivo {output_file} guardado.")

# Ejecutar la validación
if __name__ == "__main__":
    #file_path = f'Entrada_Fuentes_de_Datos/DataSet_Terminos_Negocio_Faculty360_entrada_test.xlsx'
    #file_path = f'Entrada_Fuentes_de_Datos/DataSet_Terminos_Negocio_Faculty360_entrada.xlsx'
    # Atributos SSFF
    #file_path = f'Entrada_Fuentes_de_Datos/DataSet_Terminos_Negocio_Faculty360_entrada_72_atrr_SSFF.xlsx'
    # Nueva tabla cursos SSFF
    file_path = f'Entrada_Fuentes_de_Datos/DataSet_Terminos_Negocio_Faculty360_entrada_cursos_SSFF.xlsx'
    
    inicio = datetime.datetime.now()
    print(f"🟢 Inicio del proceso: {inicio.strftime('%Y-%m-%d %H:%M:%S')}")
    validar_datos(file_path)
    fin = datetime.datetime.now()
    print(f"🔴 Fin del proceso: {fin.strftime('%Y-%m-%d %H:%M:%S')}")
    duracion = (fin - inicio).total_seconds() / 60
    print(f"⏱️ Duración total: {duracion:.2f} minutos")

🟢 Inicio del proceso: 2026-01-02 10:32:35
Procesando pestaña TAXONOMIA...
Archivo de datos inválidos detallados guardado.
Procesando pestaña CURSOS_CAP_COMPLETADOS_TC...
Archivo de datos inválidos detallados guardado.
Procesando pestaña CURSOS_CAP_COMPLETADOS_TP...
Archivo de datos inválidos detallados guardado.
Procesando pestaña CURSOS_CAP_COMPLETADOS_DA...
Archivo de datos inválidos detallados guardado.
Archivo consolidado guardado en Resultados_Perfilamiento/Perfilamiento_Consolidado.xlsx
Archivo Resultados_Perfilamiento/Perfilamiento_CURSOS_CAP_COMPLETADOS_TC.xlsx guardado.
Archivo Resultados_Perfilamiento/Perfilamiento_CURSOS_CAP_COMPLETADOS_TP.xlsx guardado.
Archivo Resultados_Perfilamiento/Perfilamiento_CURSOS_CAP_COMPLETADOS_DA.xlsx guardado.
🔴 Fin del proceso: 2026-01-02 11:09:18
⏱️ Duración total: 36.73 minutos
